In [2]:
import os
from dotenv import load_dotenv
from langchain_classic.schema import embeddings
from langchain_community.document_loaders import PyPDFLoader
from streamlit import success
from transformers.models.rag import retrieval_rag
from langchain.chat_models import init_chat_model

load_dotenv()


C:\Users\User NA\AppData\Local\Temp\ipykernel_19472\106302753.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [3]:


book_path = "D:/langchain/telecom_guide.pdf"
loader = PyPDFLoader(book_path)
pages = loader.load()
print(f"loaded {len(pages)} pages")
print(pages[1].page_content[:500])

loaded 9 pages
Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size= 600,
    chunk_overlap= 100,
    separators=["\n \n","\n","."," "],
)
chunks = splitter.split_documents(pages)
len(chunks)

37

In [5]:
chunks[1].page_content[:]

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app'

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name ="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
print(f"the data is stored{vector_store._chroma_collection.count()}succesful!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the data is stored37succesful!


In [7]:
retrieval_rag = vector_store.as_retriever(search_kwargs={"k":3})
test_query =" what is volte"
retrived = retrieval_rag.invoke(test_query)

for i,doc in enumerate(retrived,1):

 print(f" --- Chunk {i} --- ")
 print(doc.page_content[:300])
 print()

 --- Chunk 1 --- 
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

 --- Chunk 2 --- 
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

 --- Chunk 3 --- 
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

SYSTEM_PROMPT = """
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

def format_docs(docs):
    return "\n\n" "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate. from_messages([
("system", SYSTEM_PROMPT),
("human", "{question}"),
])

llm = init_chat_model("groq:llama-3.3-70b-versatile")


chain = (
    {"context": retrieval_rag | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain assembled.")

RAG chain assembled.


In [9]:
question = "How does international roaming work and what charges should I expect?"
print(f"question:, {question}\n")
print("answer",chain.invoke(question))

question:, How does international roaming work and what charges should I expect?

answer When you travel outside your home network's coverage area, your device connects to a partner network in the visited country, which is called roaming. The visited network authenticates you through a signaling protocol, and your home network validates your subscription and authorizes service. All your data, voice, and SMS traffic is then tunnelled back to your home network for billing, which can add some latency.

As for charges, our network is divided into three roaming zones based on agreements and cost structures:

* Zone A (EU, UK, Australia, New Zealand) has the lowest roaming rates.
* Zone B (USA, Canada, Japan, Singapore) has moderate rates.
* Zone C (Rest of World) has the highest per-MB and per-minute charges.

To avoid bill shock, it's recommended that you purchase a roaming bundle before traveling to Zone B or C countries. If you use data before purchasing a bundle, you'll be charged at st